# Chapter 8: Knowledge Distillation — Making Powerful Models Practical

This chapter covers:
- Distilling knowledge from large models
- Using temperature-scaled soft targets
- Building DeepSeek-R1's distilled models

In this chapter, we take everything our massive DeepSeek-R1 model has learned and compress it into models small enough to run on a single consumer GPU or even a phone. The key technique is **knowledge distillation** — training a small "student" model to reproduce the behavior of a larger "teacher" model.

## Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

---

## The Teacher-Student Paradigm

Instead of training a small model from scratch on raw data, we train it to **mimic a larger, more capable model**. The large model becomes the **teacher**, and the small model becomes the **student**.

### Why Soft Labels Carry More Information Than Hard Labels

A **hard label** is a one-hot vector: `[1, 0, 0, 0]` — approximately 1 bit of information per sample.

A **soft label** from the teacher: `[cat: 86%, dog: 12%, fox: 1.5%, bird: 0.5%]` — communicates a rich web of inter-class relationships. This is what Geoffrey Hinton called **"dark knowledge"** in his seminal 2015 paper.

---

## Temperature and Dark Knowledge

Under a standard softmax (T=1), the teacher's output is too peaked — dark knowledge is hidden in near-zero probabilities. **Temperature scaling** solves this by dividing logits by T before softmax:

$$q_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **T=1**: Standard softmax (too peaked)
- **T=2–5**: The Goldilocks zone (dark knowledge emerges)
- **T≥10**: Too soft (signal lost in near-uniformity)

### Listing 8.1: Visualizing Temperature-Scaled Softmax

In [ ]:
def visualize_temperature(logits, temps):
    """Show how temperature affects softmax."""
    for T in temps:
        probs = F.softmax(logits / T, dim=-1)
        print(f"T={T:2d}: {[f'{p:.3f}' for p in probs.tolist()]}")

logits = torch.tensor([5.0, 3.0, 1.0, -1.0])
visualize_temperature(logits, [1, 2, 3, 5, 10])

Running this code shows the full progression from peaked to uniform:

```
T= 1: [0.865, 0.117, 0.016, 0.002]
T= 2: [0.657, 0.242, 0.079, 0.022]
T= 3: [0.523, 0.268, 0.138, 0.071]
T= 5: [0.421, 0.286, 0.183, 0.110]
T=10: [0.329, 0.270, 0.221, 0.181]
```

At **T=3**, the dark knowledge is clearly visible:
- "Dog" (26.8%) — the teacher sees dogs as somewhat cat-like
- "Fox" (13.8%) — foxes share some visual features with cats
- "Bird" (7.1%) — birds are the least similar to cats

### Why T² Scaling Matters

When we divide logits by T, the resulting gradients are scaled down by $1/T^2$. To compensate, we multiply the soft loss by $T^2$:

$$\mathcal{L}_{\text{soft}} = T^2 \cdot \text{KL}\big(\text{softmax}(z_t / T) \,\|\, \text{softmax}(z_s / T)\big)$$

---

## Building the Distillation Loss: From Naive to Powerful

### The Three Attempts

1. **Attempt #1**: Train from scratch with hard labels — ~1 bit of info per sample, slow learning
2. **Attempt #2**: Match teacher's output at T=1 — still too peaked, marginal improvement
3. **Attempt #3**: Temperature-scaled soft targets + hard labels — **the complete solution**

### The Complete Distillation Loss

The final loss has two components:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{hard}} + (1 - \alpha) \cdot T^2 \cdot \text{KL}(\text{teacher\_soft} \,\|\, \text{student\_soft})$$

- **$\mathcal{L}_{\text{hard}}$**: Standard cross-entropy with ground-truth labels (grounding)
- **$\mathcal{L}_{\text{soft}}$**: KL divergence between teacher and student soft targets (knowledge transfer)
- **$\alpha$**: Controls the balance (typically 0.3 for hard, 0.7 for soft)
- **$T^2$**: Compensates for reduced gradient magnitude at higher temperatures

### Listing 8.2: The Knowledge Distillation Loss Function

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels,
                      temperature=3.0, alpha=0.3):
    """Complete knowledge distillation loss combining hard and soft targets."""
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    kl_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean')

    hard_loss = F.cross_entropy(student_logits, labels)

    return alpha * hard_loss + (1 - alpha) * temperature**2 * kl_loss

Let's verify the loss function with a quick test:

In [ ]:
# Quick test of the distillation loss
torch.manual_seed(42)
student_logits = torch.randn(4, 10)  # batch=4, num_classes=10
teacher_logits = torch.randn(4, 10)
labels = torch.randint(0, 10, (4,))

loss = distillation_loss(student_logits, teacher_logits, labels, temperature=3.0, alpha=0.3)
print(f"Distillation loss: {loss.item():.4f}")

# Compare components
hard_only = F.cross_entropy(student_logits, labels)
print(f"Hard loss only:    {hard_only.item():.4f}")
print(f"\nThe distillation loss combines hard labels (grounding) with")
print(f"soft targets (dark knowledge transfer).")

---

## DeepSeek-R1's Distillation Recipe

DeepSeek took a fundamentally different approach from classical KD:

| | Classical KD | DeepSeek's Approach |
|---|---|---|
| **Knowledge source** | Teacher's logit distribution | Teacher's generated text |
| **Loss function** | KL divergence + temperature | Standard cross-entropy (SFT) |
| **What transfers** | Per-token probability distributions | Complete reasoning strategies |
| **Teacher access** | White-box (logits needed) | Black-box (text output only) |

### The 800K Training Dataset

DeepSeek curated ~804,745 samples via **rejection sampling**:
- Math: 395K samples (49%)
- Code: 211K samples (26%)
- General: 178K samples (22%)
- STEM + Logic: ~20K samples (3%)

For each prompt, multiple responses were generated, verified for correctness, and filtered for quality. Average sample length: **5,355 tokens** — these are detailed chain-of-thought reasoning traces.

### The Six Distilled Models

| Model | Base | Params | AIME 2024 | MATH-500 |
|-------|------|--------|-----------|----------|
| DeepSeek-R1 (Teacher) | DeepSeek-V3 | 671B | 79.8% | 97.3% |
| R1-Distill-Qwen-32B | Qwen2.5-32B | 32B | 72.6% | 94.3% |
| R1-Distill-Qwen-14B | Qwen2.5-14B | 14B | 69.7% | 93.9% |
| R1-Distill-Llama-70B | Llama-3.3-70B | 70B | 70.0% | 94.5% |
| R1-Distill-Qwen-7B | Qwen2.5-7B | 7B | 55.5% | 92.8% |
| R1-Distill-Llama-8B | Llama-3.1-8B | 8B | 50.4% | 89.1% |
| R1-Distill-Qwen-1.5B | Qwen2.5-1.5B | 1.5B | 28.9% | 83.9% |

---

## Implementing Classical Knowledge Distillation in PyTorch

Now we implement a complete, runnable distillation pipeline using CIFAR-10.

### The Teacher and Student Architectures

- **Teacher**: A larger CNN (3 conv layers, 512-unit FC layer) — ~435K parameters
- **Student**: A smaller CNN (2 conv layers, 128-unit FC layer) — ~46K parameters
- **Compression ratio**: ~9.5x

### Listing 8.3: The DistillationTrainer Class

In [ ]:
class TeacherCNN(nn.Module):
    """Larger CNN for CIFAR-10 classification (the teacher)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class StudentCNN(nn.Module):
    """Smaller CNN for CIFAR-10 classification (the student)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [ ]:
class DistillationTrainer:
    """Manages the complete knowledge distillation pipeline."""

    def __init__(self, teacher, student, temperature=3.0, alpha=0.3, lr=1e-3):
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.device = next(student.parameters()).device
        self.optimizer = torch.optim.Adam(student.parameters(), lr=lr)

        # Freeze the teacher — it is a stationary compass
        self.teacher.eval()
        for param in self.teacher.parameters():
            param.requires_grad = False

    def train_step(self, images, labels):
        """One training step: forward pass through both models, compute loss, update student."""
        self.student.train()
        images, labels = images.to(self.device), labels.to(self.device)

        # Teacher forward pass (no gradients needed)
        with torch.no_grad():
            teacher_logits = self.teacher(images)

        # Student forward pass
        student_logits = self.student(images)

        # Compute the combined distillation loss
        loss = distillation_loss(
            student_logits, teacher_logits, labels,
            temperature=self.temperature, alpha=self.alpha
        )

        # Update student weights
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()

    def train_epoch(self, dataloader):
        """Train for one full epoch."""
        total_loss = 0.0
        for images, labels in dataloader:
            total_loss += self.train_step(images, labels)
        return total_loss / len(dataloader)

### Listing 8.4 & 8.5: Running Distillation on CIFAR-10

The complete pipeline:
1. Load CIFAR-10 data
2. Train the teacher model
3. Distill the teacher's knowledge into the student
4. Train a baseline student from scratch for comparison
5. Evaluate all three models

In [ ]:
def evaluate(model, dataloader, device):
    """Evaluate a model on a dataset and return accuracy."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


def train_from_scratch(model, dataloader, epochs, device, lr=1e-3):
    """Train a model from scratch using only hard labels."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    return model

In [ ]:
def run_distillation():
    """Complete distillation pipeline on CIFAR-10."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # --- Data ---
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform)

    trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

    # --- Step 1: Train the Teacher ---
    print("\n" + "=" * 50)
    print("Step 1: Training the Teacher")
    print("=" * 50)
    teacher = TeacherCNN().to(device)
    teacher_params = sum(p.numel() for p in teacher.parameters())
    print(f"Teacher parameters: {teacher_params:,}")
    teacher = train_from_scratch(teacher, trainloader, epochs=20, device=device)
    teacher_acc = evaluate(teacher, testloader, device)
    print(f"Teacher test accuracy: {teacher_acc:.2f}%")

    # --- Step 2: Distill into the Student ---
    print("\n" + "=" * 50)
    print("Step 2: Distilling Knowledge into the Student")
    print("=" * 50)
    distilled_student = StudentCNN().to(device)
    student_params = sum(p.numel() for p in distilled_student.parameters())
    print(f"Student parameters: {student_params:,}")
    print(f"Compression ratio: {teacher_params / student_params:.1f}x")

    trainer = DistillationTrainer(
        teacher=teacher, student=distilled_student,
        temperature=3.0, alpha=0.3, lr=1e-3
    )

    for epoch in range(20):
        avg_loss = trainer.train_epoch(trainloader)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/20, Loss: {avg_loss:.4f}")

    distilled_acc = evaluate(distilled_student, testloader, device)
    print(f"Distilled student test accuracy: {distilled_acc:.2f}%")

    # --- Step 3: Train a Baseline Student from Scratch ---
    print("\n" + "=" * 50)
    print("Step 3: Training Baseline Student (from scratch)")
    print("=" * 50)
    baseline_student = StudentCNN().to(device)
    baseline_student = train_from_scratch(baseline_student, trainloader, epochs=20, device=device)
    baseline_acc = evaluate(baseline_student, testloader, device)
    print(f"Baseline student test accuracy: {baseline_acc:.2f}%")

    # --- Results ---
    print("\n" + "=" * 50)
    print("RESULTS SUMMARY")
    print("=" * 50)
    print(f"{'Model':<30} {'Params':>10} {'Accuracy':>10}")
    print("-" * 50)
    print(f"{'Teacher (large CNN)':<30} {teacher_params:>10,} {teacher_acc:>9.2f}%")
    print(f"{'Student (from scratch)':<30} {student_params:>10,} {baseline_acc:>9.2f}%")
    print(f"{'Student (distilled)':<30} {student_params:>10,} {distilled_acc:>9.2f}%")
    print(f"\nDistillation improvement: {distilled_acc - baseline_acc:+.2f}% over baseline")
    print(f"Teacher-student gap:     {teacher_acc - distilled_acc:.2f}%")


run_distillation()

### Expected Results

After running the distillation pipeline, you should see results similar to:

```
RESULTS SUMMARY
==================================================
Model                             Params   Accuracy
--------------------------------------------------
Teacher (large CNN)              ~435,000     ~78%
Student (from scratch)            ~46,000     ~70%
Student (distilled)               ~46,000     ~73%

Distillation improvement: +3% over baseline
```

The distilled student, with **~9.5x fewer parameters**, closes the gap with the teacher by learning from the teacher's dark knowledge. The ~3% improvement over the baseline demonstrates that the soft targets carry significantly more information than hard labels alone.

---

## Conclusion

In this chapter, we explored knowledge distillation — the technique that makes powerful models practical:

1. **Dark knowledge** — The teacher's soft probability distribution contains rich inter-class similarity information invisible in hard labels
2. **Temperature scaling** — Raising temperature reveals this dark knowledge; the Goldilocks zone is typically T=2–5
3. **The distillation loss** — Combines hard-label grounding ($\alpha$) with soft-target knowledge transfer ($T^2 \cdot \text{KL}$)
4. **DeepSeek's approach** — For reasoning models, chain-of-thought distillation via SFT on curated teacher outputs proves more effective than classical logit-matching
5. **The results** — DeepSeek-R1-Distill-Qwen-1.5B (0.2% of teacher size) beats GPT-4o on mathematical reasoning

Knowledge distillation is the bridge between what the most powerful models can do and what can actually be deployed in the real world. It is arguably one of the most important techniques in modern AI, enabling AI reasoning to have broad impact beyond the data centers where the largest models reside.